# **netflix Data Insights**

## Objectives

* perform ETL processes on Netflix Movies and Shows, engineer meaningful features and derive insights through data visualizations.

## Inputs

* Write down which data or information you need to run the notebook 

## Outputs

* Write here which files, code or artefacts you generate by the end of the notebook 

## Additional Comments

* If you have any additional comments that don't fit in the previous bullets, please state them here. 



---

# Change working directory

* We are assuming you will store the notebooks in a subfolder, therefore when running the notebook in the editor, you will need to change the working directory

We need to change the working directory from its current folder to its parent folder
* We access the current directory with os.getcwd()

In [1]:
import os
current_dir = os.getcwd()
current_dir

'c:\\Users\\ovrcm\\OneDrive\\Documents\\vscode-projects\\netflix-data-analysis\\jupyter_notebooks'

We want to make the parent of the current directory the new current directory
* os.path.dirname() gets the parent directory
* os.chir() defines the new current directory

In [2]:
os.chdir(os.path.dirname(current_dir))
print("You set a new current directory")

You set a new current directory


Confirm the new current directory

In [3]:
current_dir = os.getcwd()
current_dir

'c:\\Users\\ovrcm\\OneDrive\\Documents\\vscode-projects\\netflix-data-analysis'

# import libaries and packages

In [30]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

---

### 1. Load the Dataset

In this step, I load the Netflix Movies and TV Shows dataset into a pandas DataFrame.  
This allows me to inspect the structure of the data and begin the ETL (Extract, Transform, Load) process.  
I also display the first few rows to confirm the dataset has loaded correctly.

In [31]:
df = pd.read_csv('../dataset/RawData/netflix_titles.csv')
df.head(5)

,show_id,type,title,director,cast,country,date_added,release_year,rating,duration,listed_in,description
0,s1,Movie,Dick Johnson Is Dead,Kirsten Johnson,NaN,United States,"September 25, 2021",2020,PG-13,90 min,Documentaries,"As her father nears the end of his life, filmm..."
1,s2,TV Show,Blood & Water,NaN,"Ama Qamata, Khosi Ngema, Gail Mabalane, Thaban...",South Africa,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, TV Dramas, TV Mysteries","After crossing paths at a party, a Cape Town t..."
2,s3,TV Show,Ganglands,Julien Leclercq,"Sami Bouajila, Tracy Gotoas, Samuel Jouy, Nabi...",NaN,"September 24, 2021",2021,TV-MA,1 Season,"Crime TV Shows, International TV Shows, TV Act...",To protect his family from a powerful drug lor...
3,s4,TV Show,Jailbirds New Orleans,NaN,NaN,NaN,"September 24, 2021",2021,TV-MA,1 Season,"Docuseries, Reality TV","Feuds, flirtations and toilet talk go down amo..."
4,s5,TV Show,Kota Factory,NaN,"Mayur More, Jitendra Kumar, Ranjan Raj, Alam K...",India,"September 24, 2021",2021,TV-MA,2 Seasons,"International TV Shows, Romantic TV Shows, TV ...",In a city of coaching centers known to train I...


### 2. Inspect Dataset Structure

Here I examine the dataset's structure using `df.info()` and check for missing values using `df.isnull().sum()`.  
This helps identify:
- Data types  
- Columns requiring cleaning  
- Missing or inconsistent data  
- Potential transformations needed before analysis


In [32]:
df.info()
df.isnull().sum()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8807 entries, 0 to 8806
Data columns (total 12 columns):
 #   Column        Non-Null Count  Dtype 
---  ------        --------------  ----- 
 0   show_id       8807 non-null   object
 1   type          8807 non-null   object
 2   title         8807 non-null   object
 3   director      6173 non-null   object
 4   cast          7982 non-null   object
 5   country       7976 non-null   object
 6   date_added    8797 non-null   object
 7   release_year  8807 non-null   int64 
 8   rating        8803 non-null   object
 9   duration      8804 non-null   object
 10  listed_in     8807 non-null   object
 11  description   8807 non-null   object
dtypes: int64(1), object(11)
memory usage: 825.8+ KB


show_id            0
type               0
title              0
director        2634
cast             825
country          831
date_added        10
release_year       0
rating             4
duration           3
listed_in          0
description        0
dtype: int64

### 3. Handle Missing Values

The dataset inspection shows several columns with missing values, including `director`, `cast`, `country`, `rating`, and `duration`.  
These missing values are expected because Netflix does not always provide full metadata for every title.

Instead of dropping rows (which would remove useful data and reduce the dataset size), I replace missing values with `"Unknown"`.  
This approach preserves all entries while still allowing meaningful analysis in later steps.

In [33]:
df['director'] = df['director'].fillna('Unknown')
df['cast'] = df['cast'].fillna('Unknown')
df['country'] = df['country'].fillna('Unknown')
df['rating'] = df['rating'].fillna('Unknown')
df['duration'] = df['duration'].fillna('Unknown')
df.isnull().sum()

show_id          0
type             0
title            0
director         0
cast             0
country          0
date_added      10
release_year     0
rating           0
duration         0
listed_in        0
description      0
dtype: int64

### 4. Convert `date_added` to Datetime Format

The `date_added` column is currently stored as text (`object`).  
To analyse trends over time — such as how Netflix’s catalogue has grown — this column needs to be converted into a proper datetime format.

After converting the column, I also extract a new feature called `year_added`, which will allow me to visualise how many titles were added to Netflix each year.  
This step prepares the dataset for time‑based exploratory analysis.


In [34]:
# Convert date_added to datetime using mixed format inference
df['date_added'] = pd.to_datetime(df['date_added'], format='mixed', errors='coerce')

# Extract year_added for time-based analysis
df['year_added'] = df['date_added'].dt.year

# Preview the results
df[['date_added', 'year_added']].head()


,date_added,year_added
0,2021-09-25,2021.0
1,2021-09-24,2021.0
2,2021-09-24,2021.0
3,2021-09-24,2021.0
4,2021-09-24,2021.0


### 5. Split Genre Information

The `listed_in` column has multiple genres in one string.  
I’m splitting them into a list so I can count genres properly later and make better visualisations.


In [35]:
df['genres'] = df['listed_in'].str.split(', ')
df[['listed_in', 'genres']].head()

,listed_in,genres
0,Documentaries,[Documentaries]
1,"International TV Shows, TV Dramas, TV Mysteries","[International TV Shows, TV Dramas, TV Mysteries]"
2,"Crime TV Shows, International TV Shows, TV Act...","[Crime TV Shows, International TV Shows, TV Ac..."
3,"Docuseries, Reality TV","[Docuseries, Reality TV]"
4,"International TV Shows, Romantic TV Shows, TV ...","[International TV Shows, Romantic TV Shows, TV..."


### 6. Clean Duration Column

The `duration` column has values like "90 min" for movies and "2 Seasons" for TV shows.  
I’m going to split this into two parts:

- one column showing the number (minutes or seasons)
- one column showing whether it’s minutes or seasons

This makes it easier to analyse movie lengths and TV show season counts later.


In [36]:
df['duration_type'] = df['duration'].apply(lambda x: 'Season' if 'Season' in x else 'Minutes')

df['duration_int'] = df['duration'].str.extract(r'(\d+)').astype(float)

#preview results
df[['duration', 'duration_type', 'duration_int']].head()

,duration,duration_type,duration_int
0,90 min,Minutes,90.0
1,2 Seasons,Season,2.0
2,1 Season,Season,1.0
3,1 Season,Season,1.0
4,2 Seasons,Season,2.0


### 7. EDA: Movies vs TV Shows

Now that the data is cleaned, I’m starting the EDA.  
First, I want to see how many Movies and TV Shows are in the dataset.  
This gives me a quick idea of what type of content Netflix has more of.


In [37]:
df['type'].value_counts()


type
Movie      6131
TV Show    2676
Name: count, dtype: int64

### 8. EDA: Content Added Over Time

Now I want to see how many titles were added to Netflix each year.  
This helps me understand how the catalogue has grown over time.


In [38]:
df['year_added'].value_counts().sort_index()

year_added
2008.0       2
2009.0       2
2010.0       1
2011.0      13
2012.0       3
2013.0      11
2014.0      24
2015.0      82
2016.0     429
2017.0    1188
2018.0    1649
2019.0    2016
2020.0    1879
2021.0    1498
Name: count, dtype: int64

### 9. EDA: Ratings Distribution

Next, I want to look at the maturity ratings (like TV-MA, TV-14, PG-13).  
This helps me understand what kind of audience most Netflix content is aimed at.


In [39]:
df['rating'].value_counts()

rating
TV-MA       3207
TV-14       2160
TV-PG        863
R            799
PG-13        490
TV-Y7        334
TV-Y         307
PG           287
TV-G         220
NR            80
G             41
TV-Y7-FV       6
Unknown        4
NC-17          3
UR             3
74 min         1
84 min         1
66 min         1
Name: count, dtype: int64

### 10. EDA: Top Genres

Now I want to see which genres appear the most on Netflix.  
Since I already split the genres into lists, I can count them properly.


In [40]:
from collections import Counter
genre_counts = Counter(df['genres'].explode())
genre_counts.most_common(10)

[('International Movies', 2752),
 ('Dramas', 2427),
 ('Comedies', 1674),
 ('International TV Shows', 1351),
 ('Documentaries', 869),
 ('Action & Adventure', 859),
 ('TV Dramas', 763),
 ('Independent Movies', 756),
 ('Children & Family Movies', 641),
 ('Romantic Movies', 616)]

### 11. EDA: Top Countries

Now I want to see which countries produce the most content on Netflix.  
This gives me a quick idea of how global Netflix’s catalogue is.


In [41]:
df['country'].value_counts().head(10)

country
United States     2818
India              972
Unknown            831
United Kingdom     419
Japan              245
South Korea        199
Canada             181
Spain              145
France             124
Mexico             110
Name: count, dtype: int64

### 12. EDA: Duration Analysis (Movies vs TV Shows)

Now I want to look at the duration of movies and TV shows separately.  
Movies use minutes, and TV shows use seasons, so I’ll check the average and distribution for each.


In [42]:
# Average movie duration 
movie_duration = df[df['duration_type'] == 'Minutes']['duration_int'].describe()

# Average series duration
series_duration = df[df['duration_type'] == 'Season']['duration_int'].describe()

movie_duration, series_duration

(count    6128.000000
 mean       99.577187
 std        28.290593
 min         3.000000
 25%        87.000000
 50%        98.000000
 75%       114.000000
 max       312.000000
 Name: duration_int, dtype: float64,
 count    2676.000000
 mean        1.764948
 std         1.582752
 min         1.000000
 25%         1.000000
 50%         1.000000
 75%         2.000000
 max        17.000000
 Name: duration_int, dtype: float64)

### 13. EDA: Most Common Directors

Now I want to check which directors appear the most on Netflix.  
This helps me see which filmmakers have the most content on the platform.


In [43]:
df['director'].value_counts().head(10)


director
Unknown                   2634
Rajiv Chilaka               19
Raúl Campos, Jan Suter      18
Suhas Kadav                 16
Marcus Raboy                16
Jay Karas                   14
Cathy Garcia-Molina         13
Jay Chapman                 12
Youssef Chahine             12
Martin Scorsese             12
Name: count, dtype: int64

### 13.1 Removing "Unknown" Directors

When I looked at the most common directors, the top result was "Unknown".  
This happens because many Netflix titles in the dataset do not include a director in their metadata.  
Since "Unknown" is not a real director, keeping it in the analysis would distort the results.

To get meaningful insights, I removed "Unknown" and then looked at the actual top directors.


In [44]:
df[df['director'] != 'Unknown']['director'].value_counts().head(10)

director
Rajiv Chilaka             19
Raúl Campos, Jan Suter    18
Marcus Raboy              16
Suhas Kadav               16
Jay Karas                 14
Cathy Garcia-Molina       13
Martin Scorsese           12
Youssef Chahine           12
Jay Chapman               12
Steven Spielberg          11
Name: count, dtype: int64

### 14. EDA: Most Common Actors

Now I want to see which actors appear the most across Netflix titles.  
The `cast` column contains multiple actors separated by commas, so I need to split them and count each actor individually.


In [45]:
# Create cast_list
df['cast_list'] = df['cast'].fillna('').apply(
    lambda x: [actor.strip() for actor in x.split(',') if actor.strip()]
)

# Import Counter
from collections import Counter

# Count actors
actor_counts = Counter(df['cast_list'].explode())

# Display top 10 actors
actor_counts.most_common(10)




[('Unknown', 825),
 ('Anupam Kher', 43),
 ('Shah Rukh Khan', 35),
 ('Julie Tejwani', 33),
 ('Naseeruddin Shah', 32),
 ('Takahiro Sakurai', 32),
 ('Rupa Bhimani', 31),
 ('Akshay Kumar', 30),
 ('Om Puri', 30),
 ('Yuki Kaji', 29)]

### 14.1 Removing "Unknown" Actors

When counting the most common actors, the top result was "Unknown".  
This happens because many titles in the dataset do not include cast information,  
so the missing values were stored as "Unknown".

Since "Unknown" is not a real actor, keeping it in the analysis would distort the results.  
To get meaningful insights, I removed "Unknown" before calculating the top actors.


In [46]:

clean_actor_counts = actor_counts.copy()
clean_actor_counts.pop('Unknown', None)


clean_actor_counts.most_common(10)


[('Anupam Kher', 43),
 ('Shah Rukh Khan', 35),
 ('Julie Tejwani', 33),
 ('Naseeruddin Shah', 32),
 ('Takahiro Sakurai', 32),
 ('Rupa Bhimani', 31),
 ('Akshay Kumar', 30),
 ('Om Puri', 30),
 ('Yuki Kaji', 29),
 ('Amitabh Bachchan', 28)]

### 15. EDA: Content Over Time (Movies vs TV Shows)

Now I want to see how Netflix content has changed over time.  
I will group the data by release year and compare how many movies and TV shows were added each year.  
This helps me understand trends in Netflix’s catalogue growth.


In [49]:
content_over_time = df.groupby(['release_year', 'type']).size().unstack(fill_value=0)
content_over_time.head(20)

type,Movie,TV Show
release_year,,
1925,0,1
1942,2,0
1943,3,0
1944,3,0
1945,3,1
1946,1,1
1947,1,0
1954,2,0
1955,3,0


# Section 2

Section 2 content

---

NOTE

* You may add as many sections as you want, as long as it supports your project workflow.
* All notebook's cells should be run top-down (you can't create a dynamic wherein a given point you need to go back to a previous cell to execute some task, like go back to a previous cell and refresh a variable content)

---

# Push files to Repo

* In cases where you don't need to push files to Repo, you may replace this section with "Conclusions and Next Steps" and state your conclusions and next steps.

In [ ]:
import os
try:
  # create your folder here
  # os.makedirs(name='')
except Exception as e:
  print(e)
